In [1]:
import pandas as pd
import re
import os
from google.colab import drive

In [2]:
# 1. Setup
drive.mount('/content/drive')
input_path = '/content/drive/MyDrive/Project/data/filtered_experimental_set.csv'
output_path = '/content/drive/MyDrive/Project/data/perturbed_set.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def rename_variables(code):
    keywords = {
        'int', 'char', 'float', 'double', 'struct', 'if', 'else', 'while', 'for', 
        'return', 'break', 'continue', 'switch', 'case', 'default', 'sizeof', 
        'static', 'const', 'void', 'unsigned', 'signed', 'long', 'short', 'NULL'
    }

    # This regex matches strings in double quotes OR words
    # Group 1: Strings (to be ignored)
    # Group 2: Identifiers (to be renamed)
    pattern = r'("[^"]*")|(\b[a-zA-Z_][a-zA-Z0-9_]*\b)'

    # 1. First pass: find all valid identifiers NOT in strings
    identifiers = set()
    for match in re.finditer(pattern, code):
        if match.group(2): # If it's a word and not a string
            word = match.group(2)
            if word not in keywords and len(word) > 1:
                identifiers.add(word)

    # 2. Create mapping
    targets = sorted(list(identifiers), key=len, reverse=True)
    mapping = {old: f"var_{i+1}" for i, old in enumerate(targets)}

    # 3. Second pass: Replace only words NOT in strings
    def replace_func(match):
        if match.group(1): # If it's a string, return it untouched
            return match.group(1)
        word = match.group(2)
        return mapping.get(word, word) # Replace if in mapping, else keep

    perturbed_code = re.sub(pattern, replace_func, code)
    return perturbed_code

In [ ]:
def transform_control_flow(code):
    """
    Structural perturbation: Converts 'for(init; cond; inc) { body }' 
    into 'init; while(cond) { body; inc; }'
    """
    # Regex to capture for(init; cond; inc) { body }
    for_pattern = r'for\s*\(([^;]*);([^;]*);([^)]*)\)\s*\{'
    
    def for_to_while(match):
        init = match.group(1).strip()
        cond = match.group(2).strip()
        inc = match.group(3).strip()
        
        # Construct the while-equivalent
        # put the init before, and the increment at the start of the block 
        # (safer at start than the end for regex replacement logic)
        replacement = f"{init};\n    while({cond}) {{\n        {inc};"
        return replacement

    return re.sub(for_pattern, for_to_while, code)


In [ ]:
def transform_if_to_ternary(code):
    """
    Simplistic regex for: if(cond) { x = a; } else { x = b; } 
    -> x = cond ? a : b;
    """
    pattern = r'if\s*\(([^)]+)\)\s*{\s*(\w+)\s*=\s*([^;]+);\s*}\s*else\s*{\s*\2\s*=\s*([^;]+);\s*}'
    replacement = r'\2 = (\1) ? \3 : \4;'
    return re.sub(pattern, replacement, code)

In [ ]:
def apply_all_perturbations(code):
    code = transform_control_flow(code)     # For to While
    code = transform_if_to_ternary(code)    # If to Ternary
    code = rename_variables(code)           # Lexical renaming
    return code

In [4]:
# 2. Execution
print("--- Starting Perturbation Stage ---")
df = pd.read_csv(input_path)

# Apply both transformations
df['perturbed_code'] = df['code'].apply(apply_all_perturbations)

# Save results
df.to_csv(output_path, index=False)

print(f"--- Success! Created {len(df)} perturbed samples ---")
print(f"File saved to: {output_path}")

# 3. Quick Preview
print("\n--- Original vs Perturbed Preview ---")
print("ORIGINAL:\n", df.iloc[0]['code'][:150], "...")
print("\nPERTURBED:\n", df.iloc[0]['perturbed_code'][:150], "...")

--- Starting Perturbation Stage ---
--- Success! Created 64 perturbed samples ---
File saved to: /content/drive/MyDrive/Project/data/perturbed_set.csv

--- Original vs Perturbed Preview ---
ORIGINAL:
 ExprResolveLhs(struct xkb_context *ctx, const ExprDef *expr,
               const char **elem_rtrn,  ...

PERTURBED:
 var_1(struct var_5 *var_21, const var_14 *var_19,
               const char **var_11, const char **v ...


-------------------------


In [ ]:
import pandas as pd
import re
import random
from google.colab import drive

In [ ]:
# 1. Setup
drive.mount('/content/drive')
input_path = '/content/drive/MyDrive/Project/data/filtered_experimental_set.csv'
output_path = '/content/drive/MyDrive/Project/data/perturbed_set.csv'

In [ ]:
def transform_logic(code, index):
    """
    Applies structural changes deterministically based on the row index.
    Even rows get 'while' conversion, odd rows stay 'for'.
    """
    for_pattern = r'for\s*\(([^;]*);([^;]*);([^)]*)\)\s*\{'
    
    def for_replacer(match):
        init, cond, inc = match.groups()
        
        # Guaranteed 50/50 split based on the row number
        if index % 2 == 0: 
            return f"{init.strip()};\n    while({cond.strip()}) {{\n        {inc.strip()};"
        else:
            return match.group(0)

    code = re.sub(for_pattern, for_replacer, code)

    # Ternary is applied to all applicable rows as it is a specific logic test
    ternary_pattern = r'if\s*\(([^)]+)\)\s*{\s*(\w+)\s*=\s*([^;]+);\s*}\s*else\s*{\s*\2\s*=\s*([^;]+);\s*}'
    code = re.sub(ternary_pattern, r'\2 = (\1) ? \3 : \4;', code)

    return code

In [ ]:
def rename_variables(code):
    """
    Lexical perturbation: Renames identifiers to var_1, var_2, etc., 
    while ignoring C-style keywords and strings.
    """
    keywords = {
        'int', 'char', 'float', 'double', 'struct', 'if', 'else', 'while', 'for', 
        'return', 'break', 'continue', 'switch', 'case', 'default', 'sizeof', 
        'static', 'const', 'void', 'unsigned', 'signed', 'long', 'short', 'NULL'
    }

    # Pattern: Group 1 captures strings; Group 2 captures potential identifiers
    pattern = r'("[^"]*")|(\b[a-zA-Z_][a-zA-Z0-9_]*\b)'

    # 1. First pass: Identify variables to rename
    identifiers = set()
    for match in re.finditer(pattern, code):
        if match.group(2):  # If it's a word and not a string
            word = match.group(2)
            if word not in keywords and len(word) > 1:
                identifiers.add(word)

    # 2. Create mapping (sorted by length descending to prevent partial replacement)
    targets = sorted(list(identifiers), key=len, reverse=True)
    mapping = {old: f"var_{i+1}" for i, old in enumerate(targets)}

    # 3. Second pass: Replace only Group 2 matches found in mapping
    def replace_func(match):
        if match.group(1): 
            return match.group(1)  # Return strings untouched
        word = match.group(2)
        return mapping.get(word, word)

    return re.sub(pattern, replace_func, code)

In [ ]:
# 2. Execution
print("--- Starting Perturbation Stage ---")
df = pd.read_csv(input_path)

# Use a list comprehension or apply with lambda to pass the index
perturbed_list = []
for idx, row in df.iterrows():
    # 1. Structural change (passing the index for 50/50 consistency)
    c = transform_logic(row['code'], idx)
    # 2. Lexical change
    c = rename_variables(c)
    perturbed_list.append(c)

df['perturbed_code'] = perturbed_list

# Save
df.to_csv(output_path, index=False)
print("--- Success! Perturbations applied ---")